In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "numpy", "matplotlib", "POT"])


# Section 4.1 — Wasserstein Distance Detects Bias when Disparate Impact Fails
**Paper:** *Wasserstein Distance as a Fairness Metric*, arXiv:2102.10349v1

---

Blue College admits students from two schools under rule $r$:

| | School A | School B |
|---|---|---|
| Mechanism | Each student admitted independently w.p. $r$ | Top $r$ fraction by GPA admitted |
| Distribution | $F_1 = \delta_r$ | $F_2 = (1-r)\,\delta_0 + r\,\delta_1$ |

Both schools have $\mathbb{E}[Y]=r$, so Disparate Impact $\to 1$ — the model *appears* fair.  
Yet the distributional difference is permanent:

$$W = \sqrt{W_1(F_1,\, F_2)} \;=\; \sqrt{2r(1-r)} \;>\; 0 \qquad \forall\, r \in (0,1)$$

> **Why does DI jump around at the start?**  
> $\mathrm{DI} = \min(\hat{p}_B / \hat{p}_A,\; 1)$ is asymmetrically clipped.  
> — *Shoots up from 0:* small $r$ and small $n$ → some years $\hat{p}_A = 0$, so DI $= 0$; cumulative mean starts low then rises.  
> — *Drops from 1:* large $r$ or large $n$ → $\hat{p}_B > \hat{p}_A$ in early years (clipped to 1); once $\hat{p}_B < \hat{p}_A$ years appear the mean drifts down and stabilises.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ot

# ── Plot style ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 130, "figure.facecolor": "white",
    "axes.facecolor": "#F8F9FA", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "white", "grid.linewidth": 0.8,
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 11, "axes.labelsize": 10, "legend.fontsize": 8.5,
    "lines.linewidth": 1.3,
})

# ── Parameters ───────────────────────────────────────────────────────────────
SEED   = 42
YEARS  = 250
RULES  = [0.10, 0.25, 0.50]
N_LIST = [50, 150, 300, 1000]

GPA_MIN, GPA_MAX = 1.5, 4.0
THRESHOLDS = {r: GPA_MIN + (1 - r) * (GPA_MAX - GPA_MIN) for r in RULES}
THEORY_W   = {r: np.sqrt(2 * r * (1 - r)) for r in RULES}

RULE_COLOR = {0.10: "#E67E22", 0.25: "#27AE60", 0.50: "#2980B9"}
N_COLOR    = {50: "#E74C3C", 150: "#E67E22", 300: "#27AE60", 1000: "#2980B9"}
N_LS       = {50: (4,2), 150: (6,2,1,2), 300: (8,2), 1000: None}

# ── Core simulation ──────────────────────────────────────────────────────────
def simulate_year(rule, n, rng):
    y_A = rng.binomial(1, rule, n).astype(float)
    p_A = y_A.mean()
    y_B = (rng.uniform(GPA_MIN, GPA_MAX, n) >= THRESHOLDS[rule]).astype(float)
    p_B = y_B.mean()
    di  = min(p_B / p_A, 1.0) if p_A > 1e-9 else 0.0
    # W = sqrt(W_1) via sorted-quantile formula (O(n log n), exact for 1-D equal weights)
    w   = float(np.sqrt(np.mean(np.abs(np.full(n, rule) - np.sort(y_B)))))
    return di, w

def run_simulation(n):
    rng = np.random.default_rng(SEED)
    res = {r: {"di": np.empty(YEARS), "w": np.empty(YEARS)} for r in RULES}
    for t in range(YEARS):
        for r in RULES:
            res[r]["di"][t], res[r]["w"][t] = simulate_year(r, n, rng)
    return res

def cum_mean(a):
    return np.cumsum(a) / np.arange(1, len(a) + 1)

all_results = {n: run_simulation(n) for n in N_LIST}
print("Simulation complete.")
print("Theory W:", {r: f"{v:.4f}" for r, v in THEORY_W.items()})


## Figure 2 — Reproduction  ($n = 150$)

DI converges to 1 for all rules (model *appears* fair); $W$ stays bounded away from 0 (structural bias persists).


In [ ]:
years = np.arange(1, YEARS + 1)
n0    = 150

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("Figure 2 Reproduction  —  250 simulated years  ($n = 150$)",
             fontsize=12, fontweight="bold", y=1.01)

for ax, key, ylabel, title, ylim, extras in [
    (axes[0], "di", "Aggregate DI", "Disparate Impact", (0.68, 1.04),
     [(0.80, "#2980B9", "fair lower-bound"), (1.00, "#95A5A6", "fair upper-bound")]),
    (axes[1], "w",  "Aggregate $W$", "Wasserstein", (-0.01, 0.76),
     [(0.0, "#2980B9", "fair lower-bound")]),
]:
    for yval, col, lbl in extras:
        ax.axhline(yval, color=col, lw=1.0, ls=":", label=lbl)
    for r in RULES:
        data = cum_mean(all_results[n0][r][key]) if key == "di" \
               else all_results[n0][r][key]
        ax.plot(years, data, color=RULE_COLOR[r], lw=1.5, label=f"$r={r}$")
    if key == "w":
        for r in RULES:
            ax.axhline(THEORY_W[r], color=RULE_COLOR[r], lw=0.7, ls="--", alpha=0.6)
    ax.set_ylim(ylim)
    ax.set_xlabel("years simulated", labelpad=6)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="lower right" if key == "di" else "upper right")

fig.tight_layout()
plt.show()


## Effect of Sample Size  ($r = 0.25$)

Larger $n$ → tighter year-by-year variance in $W$; DI convergence is faster but final value unchanged.


In [ ]:
r_focus = 0.25
W_TH    = THEORY_W[r_focus]

fig, (ax_di, ax_w) = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle(rf"Effect of $n$  ($r={r_focus}$,  $W_{{\rm theory}}={W_TH:.4f}$)",
             fontsize=12, fontweight="bold", y=1.01)

ax_di.axhline(0.80, color="#2980B9", lw=1.0, ls=":", label="fair lower-bound")
ax_di.axhline(1.00, color="#95A5A6", lw=0.8, ls=":")
ax_w.axhline(W_TH, color="#2C3E50", lw=1.6, ls="--",
             label=rf"$W_{{\rm theory}}={W_TH:.4f}$", zorder=3)

for n in N_LIST:
    dashes = {"dashes": N_LS[n]} if N_LS[n] else {}
    ax_di.plot(years, cum_mean(all_results[n][r_focus]["di"]),
               color=N_COLOR[n], label=f"$n={n}$", **dashes)
    ax_w.plot(years, all_results[n][r_focus]["w"],
              color=N_COLOR[n], lw=0.8, alpha=0.85, label=f"$n={n}$", **dashes)

ax_di.set_ylim(0.68, 1.04)
ax_di.set_xlabel("years simulated"); ax_di.set_ylabel("Cumulative mean DI")
ax_di.set_title("Disparate Impact"); ax_di.legend()

ax_w.set_ylim(W_TH * 0.35, W_TH * 1.65)
ax_w.set_xlabel("years simulated"); ax_w.set_ylabel("$W$ per year")
ax_w.set_title("Wasserstein Distance"); ax_w.legend()

fig.tight_layout()
plt.show()


## Optimal Transport Internals  ($r=0.25$, $n=15$)

The cost matrix $C_{ij} = |r - Y_{B,j}|$ takes only two values: $r$ (if $Y_{B,j}=1$) or $1-r$ (if $Y_{B,j}=0$).  
The coupling $\pi^*$ concentrates mass on low-cost entries — School A students (admitted w.p. $r$) are matched to School B students with the same outcome whenever possible.


In [ ]:
VIS_R, VIS_N = 0.25, 15
rng_v  = np.random.default_rng(2024)
y_B_v  = (rng_v.uniform(GPA_MIN, GPA_MAX, VIS_N) >= THRESHOLDS[VIS_R]).astype(float)
F1_v   = np.full(VIS_N, VIS_R)
C_mat  = ot.dist(F1_v.reshape(-1,1), y_B_v.reshape(-1,1), metric="euclidean")
a = b  = np.ones(VIS_N) / VIS_N
pi_mat = ot.emd(a, b, C_mat)
W_v    = float(np.sqrt(np.sum(pi_mat * C_mat)))

col_lbl = [f"j{j}\n{'1' if y_B_v[j] else '0'}" for j in range(VIS_N)]
row_lbl = [f"i{i}" for i in range(VIS_N)]

c_vals  = f"{VIS_R} or {round(1-VIS_R, 2)}"
c_title = f"Cost matrix $C$   ($C_{{ij}} = |r - Y_{{B,j}}|$,  values: {c_vals})"
p_title = r"Optimal coupling $\pi^*$  (scaled by $n$)"

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
fig.suptitle(rf"OT Internals  ($r={VIS_R}$, $n={VIS_N}$, $W={W_v:.4f}$)",
             fontsize=12, fontweight="bold")

for ax, mat, cmap, title, cbar_lbl, is_pi in [
    (axes[0], C_mat,         "Blues",   c_title, "Cost",               False),
    (axes[1], pi_mat * VIS_N,"YlOrRd",  p_title, r"$\pi_{ij}\times n$",True),
]:
    im = ax.imshow(mat, cmap=cmap, aspect="auto",
                   vmin=0, vmax=(1.0 if not is_pi else None))
    cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cb.set_label(cbar_lbl, fontsize=9)
    ax.set_xticks(range(VIS_N)); ax.set_xticklabels(col_lbl, fontsize=7.5)
    ax.set_yticks(range(VIS_N)); ax.set_yticklabels(row_lbl, fontsize=7.5)
    ax.set_xlabel("School B student $j$  (F2 value)")
    ax.set_ylabel("School A student $i$")
    ax.set_title(title, pad=8)
    for i in range(VIS_N):
        for j in range(VIS_N):
            v = float(mat[i, j])
            if v > (0.005 if is_pi else 0.0):
                norm_v = v / (float(mat.max()) or 1)
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if norm_v > 0.55 else "#2C3E50")

fig.tight_layout()
plt.show()
